In [1]:
import pandas as pd

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv("ppr-group-24209891-train.csv")

# Get first n rows 一次只能2500行
m = 11786
n = 12500
df = df.iloc[m-1:n]

df = df.reset_index(drop=True)

import requests
import time
from tqdm import tqdm

API_KEY = "385f7a9df9ab4891ae0ca8a90f564467" # Your API Key 前往 https://opencagedata.com/ 注册后获得密钥

CSV_OUT = "data6.csv" # 输出文件命名建议顺序命名

ADDRESS_COL = "Address"
COUNTY_COL = "County"

# Create empty columns
df["lat"] = None
df["lng"] = None

def geocode(address):
    url = "https://api.opencagedata.com/geocode/v1/json"
    params = {
        "q": address,
        "key": API_KEY,
        "countrycode": "ie",
        "limit": 1
    }
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        return None, None
    
    data = response.json()
    if data["results"]:
        lat = data["results"][0]["geometry"]["lat"]
        lng = data["results"][0]["geometry"]["lng"]
        return lat, lng
    return None, None

for i in tqdm(range(len(df))):
    address = str(df.loc[i, ADDRESS_COL]) + ", " + str(df.loc[i, COUNTY_COL]) + ", Ireland"
    
    lat, lng = geocode(address)
    
    df.loc[i, "lat"] = lat
    df.loc[i, "lng"] = lng
    
    time.sleep(1)  # IMPORTANT: avoid rate limit

df.to_csv(CSV_OUT, index=False)

100%|████████████████████████████████████████████████████████████████████████████████| 715/715 [17:03<00:00,  1.43s/it]
